In [27]:
import pandas as pd

In [28]:
"""
Ad-platform naming-convention parser.
=====================================
Parses messy campaign / adset / ad names from 6 ad platforms and extracts
a canonical taxonomy. Real ad-account names have random casing, random or
absent delimiters, shuffled field order, abbreviations, typos and junk
tokens, so we use VOCABULARY MATCHING (ordered, most-specific-first) plus
a compacted-string fallback for glued tokens - NOT positional splitting.

Extracted signals:
  - country        (IT / DE)                         <- from any name
  - objective      (Awareness / Conversion)          <- mainly campaign name
  - campaign_name  (canonical campaign)              <- from campaign/ad name
  - audience       (Broad / Gift Shoppers / Retargeting) <- mainly adset name
  - creative_type  (image/video/carousel/...)        <- from ad name

Usage:
    python extract_campaign_data.py                 # runs the demo/validation
    from extract_campaign_data import parse_row     # import in Airflow/dbt-py
"""
import re
import pandas as pd


# ----------------------------------------------------------------------
# normalisation helpers
# ----------------------------------------------------------------------
def _norm(s: str) -> str:
    """lowercase; punctuation -> single spaces (keeps token boundaries)."""
    s = re.sub(r"[^a-z0-9]+", " ", str(s).lower())
    return re.sub(r"\s+", " ", s).strip()


def _compact(s: str) -> str:
    """lowercase alnum only (collapses glued tokens like 'salesbof')."""
    return re.sub(r"[^a-z0-9]+", "", str(s).lower())


# ----------------------------------------------------------------------
# vocabularies  (substring, canonical)  -- MOST SPECIFIC FIRST
# ----------------------------------------------------------------------
COUNTRY_NAMES = [
    ("deutschland", "DE"), ("germany", "DE"), ("deu", "DE"), ("ger", "DE"),
    ("italy", "IT"), ("ita", "IT"),
]
# bare 2-letter codes: matched with word-boundaries so 'de' inside
# 'evergreen' or 'it' inside 'gift' don't cause false positives
COUNTRY_CODES = [(r"\bde\b|\bd e\b", "DE"), (r"\bit\b|\bi t\b", "IT")]

OBJECTIVE = [
    ("awareness", "Awareness"), ("awrns", "Awareness"), ("awr", "Awareness"),
    ("aware", "Awareness"), ("brand", "Awareness"), ("tof", "Awareness"),
    ("upper", "Awareness"),
    ("conversion", "Conversion"), ("conv", "Conversion"), ("cvr", "Conversion"),
    ("sales", "Conversion"), ("perf", "Conversion"), ("bof", "Conversion"),
    ("lower", "Conversion"),
]

CAMPAIGN = [
    ("holiday", "Holiday Gifting"), ("hgift", "Holiday Gifting"),
    ("xmas", "Holiday Gifting"), ("gifting", "Holiday Gifting"),
    ("gift", "Holiday Gifting"),
    ("autumn", "Autumn Brand Push"), ("autmn", "Autumn Brand Push"),
    ("abp", "Autumn Brand Push"), ("brandpush", "Autumn Brand Push"),
    ("alwayson", "Always On Sales"), ("always on", "Always On Sales"),
    ("aosales", "Always On Sales"), ("ao sales", "Always On Sales"),
    ("aosale", "Always On Sales"), ("evergreen", "Always On Sales"),
    ("always sales", "Always On Sales"), ("always-sales", "Always On Sales"),
]

AUDIENCE = [
    ("giftshoppers", "Gift Shoppers"), ("giftshop", "Gift Shoppers"),
    ("gifters", "Gift Shoppers"), ("gift", "Gift Shoppers"),
    ("retarget", "Retargeting"), ("remarketing", "Retargeting"),
    ("remkt", "Retargeting"), ("rmk", "Retargeting"), ("rtgt", "Retargeting"),
    ("rtg", "Retargeting"),
    ("broad", "Broad"), ("genpop", "Broad"), ("all", "Broad"),
    ("18-44", "Broad"), ("1844", "Broad"), ("brd", "Broad"),
]

CREATIVE = [
    ("carousel", "carousel"), ("static", "static"), ("spark", "spark"),
    ("html5", "html5"), ("html", "html5"),
    ("vid", "video"), ("img", "image"), ("hdr", "responsive"),
    ("300x250", "banner"), ("728x90", "banner"), ("970x250", "banner"),
    ("1080x1080", "banner"),
]


def _match(text, compact, table, default="UNKNOWN"):
    """return canonical for first vocabulary hit (substring or compacted)."""
    for sub, val in table:
        if sub in text or sub.replace(" ", "").replace("-", "") in compact:
            return val
    return default


def _match_country(text, compact):
    for sub, val in COUNTRY_NAMES:
        if sub in text or sub in compact:
            return val
    for pat, val in COUNTRY_CODES:
        if re.search(pat, text):
            return val
    # glued bare-code fallback e.g. 'awrnsde', 'newde', 'holidaygiftde'
    if "de" in compact and "it" not in compact:
        return "DE"
    if "it" in compact and "de" not in compact:
        return "IT"
    return "UNKNOWN"


# ----------------------------------------------------------------------
# public API
# ----------------------------------------------------------------------
def parse_row(campaign_name: str, adset_name: str = "", ad_name: str = "") -> dict:
    """
    Parse the three messy names into the canonical taxonomy.
    Signals are looked for in the most-likely field first, then the others
    as fallback (a signal can leak into any name), which makes extraction
    robust to whichever field actually carries it.
    """
    cn, an, adn = _norm(campaign_name), _norm(adset_name), _norm(ad_name)
    cc, ac, adc = _compact(campaign_name), _compact(adset_name), _compact(ad_name)
    all_t = " ".join([cn, an, adn])
    all_c = "".join([cc, ac, adc])

    # country: any field
    country = _match_country(cn, cc)
    if country == "UNKNOWN":
        country = _match_country(all_t, all_c)

    # objective: campaign name first, then anywhere
    objective = _match(cn, cc, OBJECTIVE)
    if objective == "UNKNOWN":
        objective = _match(all_t, all_c, OBJECTIVE)

    # campaign: campaign name first, then ad name (carries it too)
    campaign = _match(cn, cc, CAMPAIGN)
    if campaign == "UNKNOWN":
        campaign = _match(adn + " " + an, adc + ac, CAMPAIGN)

    # audience: adset name first, then anywhere
    audience = _match(an, ac, AUDIENCE)
    if audience == "UNKNOWN":
        audience = _match(all_t, all_c, AUDIENCE)

    # creative type: ad name
    creative = _match(adn, adc, CREATIVE)

    return {
        "country": country,
        "objective": objective,
        "campaign_clean": campaign,
        "audience": audience,
        "creative_type": creative,
    }


def parse_dataframe(df: pd.DataFrame) -> pd.DataFrame:
    """apply parse_row across a dataframe, returning df + 5 clean columns."""
    parsed = df.apply(
        lambda r: parse_row(r["campaign_name"], r.get("adset_name", ""), r.get("ad_name", "")),
        axis=1, result_type="expand",
    )
    return pd.concat([df, parsed], axis=1)


# ----------------------------------------------------------------------
# validation / demo
# ----------------------------------------------------------------------
if __name__ == "__main__":
    df = pd.read_csv("~/Desktop/mrkt_pipeline/datasets/ad_spend_raw.csv")
    out = parse_dataframe(df)

    uniq = out.drop_duplicates(subset=["campaign_name", "adset_name", "ad_name"])
    fields = ["country", "objective", "campaign_clean", "audience", "creative_type"]

    print(f"Parsed {len(df):,} rows | {len(uniq)} unique name-combos\n")
    print("Resolution per field (on unique combos):")
    for c in fields:
        n = (uniq[c] == "UNKNOWN").sum()
        pct = 100 * (len(uniq) - n) / len(uniq)
        print(f"  {c:<15}: {n:>3} unknown   ({pct:.0f}% resolved)")

    print("\n--- 15 sample parsed combos ---")
    for _, r in uniq.head(15).iterrows():
        print()
        print(f"  campaign: {r['campaign_name']!r}")
        print(f"  adset:    {r['adset_name']!r}")
        print(f"  ad:       {r['ad_name']!r}")
        print(f"    => {r['country']} | {r['objective']} | {r['campaign_clean']} | {r['audience']} | {r['creative_type']}")

        out.to_csv("/Users/sabrinamarano/Desktop/mrkt_pipeline/datasets/ad_spend_cleaned.csv", index=False)
        print(f"\nSaved: ad_spend_cleaned.csv  ({out.shape[0]} rows, {out.shape[1]} cols)")
        dates = pd.to_datetime(out["date"], errors="coerce")
        out["year"] = dates.dt.year
        out["month"] = dates.dt.month

Parsed 4,834 rows | 91 unique name-combos

Resolution per field (on unique combos):
  country        :   0 unknown   (100% resolved)
  objective      :   0 unknown   (100% resolved)
  campaign_clean :   0 unknown   (100% resolved)
  audience       :   0 unknown   (100% resolved)
  creative_type  :  29 unknown   (68% resolved)

--- 15 sample parsed combos ---

  campaign: 'ITA AutumnBrandPushAwareness'
  adset:    'it -_all'
  ad:       'AUTUMN BRAND -_cr | img'
    => IT | Awareness | Autumn Brand Push | Broad | image

Saved: ad_spend_cleaned.csv  (4834 rows, 17 cols)

  campaign: 'ITA AutumnBrandPushAwareness'
  adset:    'it -_all'
  ad:       'crautumn brand||v2'
    => IT | Awareness | Autumn Brand Push | Broad | UNKNOWN

Saved: ad_spend_cleaned.csv  (4834 rows, 19 cols)

  campaign: 'copy-AUTMN BRAND  Brand.It'
  adset:    'Broad.It_reels'
  ad:       'autumn brand||creative  v1_2023'
    => IT | Awareness | Autumn Brand Push | Broad | UNKNOWN

Saved: ad_spend_cleaned.csv  (4834 r

In [29]:
df_cl = pd.read_csv("~/Desktop/mrkt_pipeline/datasets/ad_spend_cleaned.csv")
df_cl

,date,platform,campaign_id,adset_id,ad_id,campaign_name,adset_name,ad_name,spend,impressions,clicks,conversions,country,objective,campaign_clean,audience,creative_type,year,month
0,2023-10-01,Facebook,FA1001,AS50001,AD900001,ITA AutumnBrandPushAwareness,it -_all,AUTUMN BRAND -_cr | img,60.04,7600,100,3,IT,Awareness,Autumn Brand Push,Broad,image,2023,10
1,2023-10-02,Facebook,FA1001,AS50001,AD900001,ITA AutumnBrandPushAwareness,it -_all,AUTUMN BRAND -_cr | img,196.48,29358,411,4,IT,Awareness,Autumn Brand Push,Broad,image,2023,10
2,2023-10-03,Facebook,FA1001,AS50001,AD900001,ITA AutumnBrandPushAwareness,it -_all,AUTUMN BRAND -_cr | img,82.95,12762,206,7,IT,Awareness,Autumn Brand Push,Broad,image,2023,10
3,2023-10-04,Facebook,FA1001,AS50001,AD900001,ITA AutumnBrandPushAwareness,it -_all,AUTUMN BRAND -_cr | img,100.95,23315,267,9,IT,Awareness,Autumn Brand Push,Broad,image,2023,10
4,2023-10-05,Facebook,FA1001,AS50001,AD900001,ITA AutumnBrandPushAwareness,it -_all,AUTUMN BRAND -_cr | img,74.14,11842,227,2,IT,Awareness,Autumn Brand Push,Broad,image,2023,10
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4829,2023-12-27,DV360,DV1033,AS50061,AD900091,de||alwayson-conv.,RETARGET Ger||reels,alwaysonasset -_vid-01,243.66,58965,108,0,DE,Conversion,Always On Sales,Retargeting,video,2023,12
4830,2023-12-28,DV360,DV1033,AS50061,AD900091,de||alwayson-conv.,RETARGET Ger||reels,alwaysonasset -_vid-01,139.39,62554,76,0,DE,Conversion,Always On Sales,Retargeting,video,2023,12
4831,2023-12-29,DV360,DV1033,AS50061,AD900091,de||alwayson-conv.,RETARGET Ger||reels,alwaysonasset -_vid-01,76.33,25875,71,0,DE,Conversion,Always On Sales,Retargeting,video,2023,12
4832,2023-12-30,DV360,DV1033,AS50061,AD900091,de||alwayson-conv.,RETARGET Ger||reels,alwaysonasset -_vid-01,168.58,57153,206,1,DE,Conversion,Always On Sales,Retargeting,video,2023,12
